In [1]:
import json

In [2]:
def format_time(secs):
  h = int(secs // 3600)
  m = int((secs % 3600) // 60)
  s = int(secs % 60)
  cs = int(round((secs - int(secs)) * 100))
  if cs == 100:
    s += 1
    cs = 0
  return f"{h}:{m:02d}:{s:02d}.{cs:02d}"

In [3]:
def segment_to_ass(segment):
  line_start = format_time(segment["start"])
  line_end = format_time(segment["end"])
  
  ass_text = ""
  words = segment["words"]
  
  for i, w in enumerate(words):
    word_str = w["word"]
    duration = int(round((w["end"] - w["start"]) * 100))
    
    ass_text += f"{{\\kf{duration}}}{word_str}"
    
    if i < len(words) - 1:
      next_w = words[i+1]
      gap = int(round((next_w["start"] - w["end"]) * 100))
      
      if gap > 0:
        ass_text += f"{{\\kf{gap}}} "
      else:
        ass_text += "{\\kf0} "

  return f"Dialogue: 0,{line_start},{line_end},Karaoke,,0,0,0,,{ass_text}"

In [ ]:
def aligned_segments_to_ass(aligned_path):
  with open(aligned_path) as f:
    segments = json.load(f)
  ass = []

  header = """[Script Info]
  Title: Karaoke
  ScriptType: v4.00+

  [V4+ Styles]
  Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
  Style: Default,Arial,48,&H00FFFFFF,&H0000FFFF,&H00000000,&H64000000,0,0,1,2,0,2,10,10,10,1

  [Events]
  Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text

  """

  ass.append(header)

  for seg in segments:
    ass.append(segment_to_ass(seg))

  return "\n".join(ass)

In [17]:
aligned_path = "output/aligned_segments.json"
with open(aligned_path) as f:
  segments = json.load(f)

ass = []

header = """[Script Info]
Title: Karaoke
ScriptType: v4.00+

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,Arial,48,&H00FFFFFF,&H0000FFFF,&H00000000,&H64000000,0,0,1,2,0,2,10,10,10,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text

"""

ass.append(header)

for seg in segments["segments"]:
  line_start = format_time(seg["start"])
  line_end = format_time(seg["end"])
  
  ass_text = ""
  words = seg["words"]
  
  for i, w in enumerate(words):
    word_str = w["word"]
    duration = int(round((w["end"] - w["start"]) * 100))
    
    ass_text += f"{{\\kf{duration}}}{word_str}"
    
    if i < len(words) - 1:
      next_w = words[i+1]
      gap = int(round((next_w["start"] - w["end"]) * 100))
      
      if gap > 0:
        ass_text += f"{{\\kf{gap}}} "
      else:
        ass_text += "{\\kf0} "

  ass.append(f"Dialogue: 0,{line_start},{line_end},Karaoke,,0,0,0,,{ass_text}")

result = "\n".join(ass)

type(result)

str

In [ ]:
import yt_dlp
import subprocess

url = "https://www.youtube.com/watch?v=Fdv6riFxLf8"

with yt_dlp.YoutubeDL({"format": "bestvideo"}) as ydl:
  info = ydl.extract_info(url, download=False)
  video_url = info["url"]

subprocess.run([
  "ffmpeg",
  "-i", video_url,
  "-t", "120",
  "-c", "copy",
  "background.mp4"
], check=True)

[youtube] Extracting URL: https://www.youtube.com/watch?v=33kwtdW-6xY
[youtube] 33kwtdW-6xY: Downloading webpage


[youtube] 33kwtdW-6xY: Downloading android vr player API JSON


KeyboardInterrupt: 

In [2]:
import yt_dlp

url = "https://www.youtube.com/watch?v=33kwtdW-6xY"

ydl_opts = {
    "format": "bestaudio/best",
    "outtmpl": "background.%(ext)s",
    "postprocessors": [
        {
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "192",
        }
    ],
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

[youtube] Extracting URL: https://www.youtube.com/watch?v=33kwtdW-6xY
[youtube] 33kwtdW-6xY: Downloading webpage


[youtube] 33kwtdW-6xY: Downloading android vr player API JSON
[info] 33kwtdW-6xY: Downloading 1 format(s): 251
[download] Destination: background.webm
[download] 100% of    2.75MiB in 00:00:00 at 25.78MiB/s  
[ExtractAudio] Destination: background.mp3
Deleting original file background.webm (pass -k to keep)
